## Run this in a Kaggle notebook with the competition dataset attached.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error
import xgboost as xgb
from datetime import datetime

## Load data

In [ ]:
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])

## Add missing Christmas dates with sales=0

In [ ]:
families = train['family'].unique()
missing_dates = ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']
missing_dates = pd.to_datetime(missing_dates)
missing_df = pd.DataFrame({'date': missing_dates})
missing_df = missing_df.merge(pd.DataFrame({'store_nbr': range(1, 55)}), how='cross')
missing_df = missing_df.merge(pd.DataFrame({'family': families}), how='cross')
missing_df['sales'] = 0.0
missing_df['onpromotion'] = 0
train = pd.concat([train, missing_df], ignore_index=True)
train = train.sort_values(['store_nbr', 'family', 'date'])

## Concat train and test

In [ ]:
df = pd.concat([train, test], ignore_index=True)
df = df.sort_values(['store_nbr', 'family', 'date'])

## Merge stores

In [ ]:
df = df.merge(stores, on='store_nbr', how='left')

## Process oil: add lags

In [ ]:
oil = oil.sort_values('date')
oil['dcoilwtico'] = oil['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')

for lag in [1, 3, 7, 14]:
    oil[f'oil_lag_{lag}'] = oil['dcoilwtico'].shift(lag)

oil = oil.fillna(oil['dcoilwtico'].mean())
df = df.merge(oil, on='date', how='left')


## Process holidays: exclude transferred, merge by locale

In [ ]:
# Keep only non-transferred holidays
holidays = holidays[holidays['transferred'] == False]

# Rename columns for clarity
holidays = holidays.rename(
    columns={
        'type': 'holiday_type',
        'locale_name': 'holiday_locale_name',
        'description': 'holiday_description'
    }
)

# Split holidays by locale
national_holidays = holidays[holidays['locale'] == 'National']
regional_holidays = holidays[holidays['locale'] == 'Regional']
local_holidays = holidays[holidays['locale'] == 'Local']

# Merge national holidays (on date)
df = df.merge(
    national_holidays[['date', 'holiday_type', 'holiday_description']],
    on='date',
    how='left',
    suffixes=('', '_national')
)

# Merge regional holidays (match on date + state)
# In the dataset, 'holiday_locale_name' holds the state name
df = df.merge(
    regional_holidays.rename(columns={'holiday_locale_name': 'state'})[
        ['date', 'holiday_type', 'holiday_description', 'state']
    ],
    on=['date', 'state'],
    how='left',
    suffixes=('', '_regional')
)

# Merge local holidays (on date, city)
df = df.merge(
    local_holidays.rename(columns={'holiday_locale_name': 'city'})[
        ['date', 'holiday_type', 'holiday_description', 'city']
    ],
    on=['date', 'city'],
    how='left'
)

# Rename local columns to make them explicit
df = df.rename(
    columns={
        'holiday_type': 'holiday_type_local',
        'holiday_description': 'holiday_description_local'
    }
)


## Fill NaN in holiday columns

In [ ]:
cols_to_fill = [
    'holiday_type_national',
    'holiday_description_national',
    'holiday_type_regional',
    'holiday_description_regional',
    'holiday_type_local',
    'holiday_description_local'
]

for col in cols_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna('None')


## Add time features

In [ ]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['day_of_year'] = df['date'].dt.dayofyear
df['is_payday'] = ((df['date'].dt.day == 15) | (df['date'].dt.day == df['date'].dt.days_in_month)).astype(int)

## Add sinusoidal features for seasonality

In [ ]:
for period in [7, 30.5, 365.25]:
    df[f'sin_{period}'] = np.sin(2 * np.pi * df['day_of_year'] / period)
    df[f'cos_{period}'] = np.cos(2 * np.pi * df['day_of_year'] / period)


## Add earthquake feature

In [ ]:
earthquake_date = datetime(2016, 4, 16)
df['days_since_earthquake'] = (df['date'] - earthquake_date).dt.days.clip(lower=0)
df['earthquake_effect'] = np.where(df['days_since_earthquake'] > 0, np.minimum(df['days_since_earthquake'], 30) / 30, 0)

## Add lag and rolling features for sales (per store-family)

In [ ]:
lags = [1, 7, 14, 21, 28]
for lag in lags:
    df[f'sales_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)

windows = [7, 14, 28]
for window in windows:
    df[f'sales_rolling_mean_{window}'] = (
        df.groupby(['store_nbr', 'family'])['sales']
        .rolling(window, min_periods=1)
        .mean()
        .shift(1)
        .reset_index(level=[0, 1], drop=True)
    )


## Fill NaN in lag/rolling with 0 (for early dates)

In [ ]:
lag_roll_cols = [f'sales_lag_{lag}' for lag in lags] + [f'sales_rolling_mean_{window}' for window in windows]
df[lag_roll_cols] = df[lag_roll_cols].fillna(0)

## Prepare for modeling

In [ ]:
categorical_cols = ['store_nbr', 'family', 'city', 'state', 'type', 'cluster',
'holiday_type_national', 'holiday_type_regional', 'holiday_type_local']

## But since training per family, family is not in X

In [ ]:
submission = pd.DataFrame()
families = df['family'].unique()

for family in families:
    print(f"Processing family: {family}")
    df_family = df[df['family'] == family].copy()


## Split train/test

In [ ]:
train_family = df_family[df_family['sales'].notnull()]
test_family = df_family[df_family['sales'].isnull()]

## Features and target

In [ ]:
drop_cols = ['id', 'date', 'sales', 'family', 'holiday_description_national', 'holiday_description_regional', 'holiday_description_local']  # Drop descriptions to avoid high cardinality
X = train_family.drop(columns=drop_cols, errors='ignore')
y = np.log1p(train_family['sales'])
X_test = test_family.drop(columns=drop_cols, errors='ignore')

## Label encode categoricals (excluding family since per model)

In [ ]:
cat_cols = [col for col in categorical_cols if col != 'family' and col in X.columns]

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined.astype(str))
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))


## Train XGBoost

In [ ]:
from sklearn.model_selection import train_test_split

# --- Ensure object columns are categorical ---
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = X_test[col].astype('category')

# --- Save test IDs ---
test_ids = X_test.index if 'id' not in X_test.columns else X_test['id']

# --- Model definition ---
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',  # or 'gpu_hist' if GPU available
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse',
    enable_categorical=True
)

# --- Train model ---
model.fit(X, y)

# --- Predictions ---
y_pred = model.predict(X_test)

# --- Create submission ---
submission = pd.DataFrame({
    'id': test_ids,
    'sales': y_pred
}).reset_index(drop=True)

print(submission.head())


## Predict

In [ ]:
pred_log = model.predict(X_test)
pred = np.expm1(pred_log)
pred = np.clip(pred, 0, None)

## Append to submission

In [ ]:
temp_sub = pd.DataFrame({'id': test_family['id'], 'sales': pred})
submission = pd.concat([submission, temp_sub])

## Sort submission by id and save

In [ ]:
submission = submission.sort_values('id')
submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")